# Explore LOF anomalies

In [1]:
import os
# Set environment variables to disable multithreading
# as users will probably want to set the number of cores
# to the max of their computer.
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

In [3]:
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.preprocessing import StandardScaler
import umap

from sdss.metadata import MetaData

meta = MetaData()

# Custom Functions

## Winner LOF

In [35]:
def pick_lof_params(df, consensus_threshold):

    """
    Given a dataframe of boolean anomaly flags from different
    LOF models (with different hyperparameters), calculate
    which parameter set is the most stable, i.e. which
    parameter set's anomalies overlap the most with the
    consensus anomalies (those that at least `consensus_threshold`
    models agreed were anomalies).
    """

    assert consensus_threshold > 0
    assert consensus_threshold <= df.shape[1] - 1

    stability_scores = {}

    # Identify points that at least 50% of models agreed were anomalies
    consensus_anomalies = df['consensus_score'] >= consensus_threshold

    for col in df.columns[:-1]: # Exclude the consensus_score column
        # How many of this model's anomalies are also 'consensus' anomalies?
        overlap = (df[col] & consensus_anomalies).sum()
        stability_scores[col] = overlap

    # The "Optimal" params are the ones with the highest overlap
    best_params = max(stability_scores, key=stability_scores.get)

    print(f"The most stable parameter set is: {best_params}")

    return stability_scores, best_params

## Scale data

In [4]:
def standard_scaler(latent_arr):

    scaler = StandardScaler()
    
    latent_scaled = scaler.fit_transform(latent_arr)
    
    return latent_scaled

# Config

## Directories

In [6]:
phd_dir = "/home/elom/phd"
thesis_dir = f"{phd_dir}/thesis"
ch4_dir = f"{thesis_dir}/chapters/04_figures"
data_dir = f"{phd_dir}/code"
spectra_dir = f"{data_dir}/spectra"
models_dir = f"{data_dir}/models"
latent_dir = f"{data_dir}/latent"
bins_ids = [f'bin_{i:02d}' for i in range(4)] 

## Data

In [7]:
wave = np.load(f"{spectra_dir}/wave_spectra_imputed.npy")
wave_nm = wave*0.1
n_wave = wave.shape

spectra = np.load(
    f"{spectra_dir}/spectra_imputed.npy",
    mmap_mode="r"
)

final_meta_df = pd.read_csv(
    f"{spectra_dir}/final_spec_n_z_warning_drop.csv.gz",
    index_col="specobjid",
)

idx_id_spec = np.load(
    f"{spectra_dir}/ids_imputing.npy",
    mmap_mode='r'
)


## Latent per bin

In [8]:
latent_bin_dict = {}

for bin_id in bins_ids:

    latent_bin_dict[bin_id] = np.load(
        f"{latent_dir}/{bin_id}/latent_{bin_id}.npy"
    )

In [46]:
lof_hyper_params_df_dict = {}

for bin_id in ['bin_02', 'bin_03']: # bins_ids:

    print(f"Loading LOF hyperparameter search results for {bin_id}")
    lof_hyper_params_df_dict[bin_id] = pd.read_csv(
        f"{latent_dir}/{bin_id}/lof_hypersearch_{bin_id}.csv",
    )

Loading LOF hyperparameter search results for bin_02
Loading LOF hyperparameter search results for bin_03


In [50]:
stability_scores_dict = {}
best_params_dict = {}
consensus_threshold = 7

for bin_id, df in lof_hyper_params_df_dict.items():

    print(f"Calculating stability scores for {bin_id}")

    stability_scores, best_params = pick_lof_params(
        df=df.copy(),
        consensus_threshold=consensus_threshold
    )

    stability_scores_dict[bin_id] = stability_scores
    best_params_dict[bin_id] = best_params

Calculating stability scores for bin_02
The most stable parameter set is: n60_euclidean
Calculating stability scores for bin_03
The most stable parameter set is: n60_euclidean


In [48]:
stability_scores_dict

{'bin_02': {'n20_euclidean': 1327,
  'n20_manhattan': 1335,
  'n20_cosine': 114,
  'n40_euclidean': 1480,
  'n40_manhattan': 1482,
  'n40_cosine': 116,
  'n60_euclidean': 1505,
  'n60_manhattan': 1499,
  'n60_cosine': 120,
  'n80_euclidean': 1489,
  'n80_manhattan': 1482,
  'n80_cosine': 120,
  'n100_euclidean': 1457,
  'n100_manhattan': 1450,
  'n100_cosine': 121},
 'bin_03': {'n20_euclidean': 1359,
  'n20_manhattan': 1373,
  'n20_cosine': 153,
  'n40_euclidean': 1509,
  'n40_manhattan': 1508,
  'n40_cosine': 168,
  'n60_euclidean': 1546,
  'n60_manhattan': 1533,
  'n60_cosine': 169,
  'n80_euclidean': 1533,
  'n80_manhattan': 1513,
  'n80_cosine': 168,
  'n100_euclidean': 1518,
  'n100_manhattan': 1493,
  'n100_cosine': 171}}

In [49]:
lof_hyper_params_df_dict['bin_03'][['n60_euclidean', 'consensus_score']].sum()

n60_euclidean       1819
consensus_score    27285
dtype: int64